In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import ipywidgets as widgets
from IPython.display import display, clear_output

In [10]:

# Load the data
df = pd.read_csv('/content/propertydataset.csv')

# Clean the data
df = df.dropna(subset=['Town', 'List Year', 'Property Type', 'Sale Amount'])
df['List Year'] = df['List Year'].astype(int)
df['Sale Amount'] = df['Sale Amount'].astype(float)

# Define features and target
X = df[['Town', 'List Year', 'Property Type']]
y = df['Sale Amount']

# Create preprocessing for categorical features
categorical_features = ['Town', 'Property Type']
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Create preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough')

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a pipeline with preprocessing and model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

In [11]:
# Train the model
model.fit(X_train, y_train)

# Get unique towns and property types for dropdowns
towns = sorted(df['Town'].unique())
property_types = sorted(df['Property Type'].unique())
years = sorted(df['List Year'].unique())

# Create interactive widgets
town_dropdown = widgets.Dropdown(
    options=towns,
    description='Town:',
    value=towns[0]
)

year_slider = widgets.IntSlider(
    value=max(years),
    min=min(years),
    max=max(years)+5,  # Allow predictions for future years
    step=1,
    description='Year:',
    continuous_update=False
)

property_type_dropdown = widgets.Dropdown(
    options=property_types,
    description='Type:',
    value=property_types[0]
)

output = widgets.Output()

def predict_and_display(change):
    with output:
        clear_output()
        town = town_dropdown.value
        list_year = year_slider.value
        property_type = property_type_dropdown.value

        input_data = pd.DataFrame({
            'Town': [town],
            'List Year': [list_year],
            'Property Type': [property_type]
        })

        prediction = model.predict(input_data)[0]
        print(f"Predicted sale amount for a {property_type} property in {town} listed in {list_year}: ${prediction:,.2f}")

        # Display some historical data for context if available
        town_data = df[(df['Town'] == town) & (df['Property Type'] == property_type)]
        if not town_data.empty:
            avg_price = town_data['Sale Amount'].mean()
            print(f"Historical average sale amount for {property_type} properties in {town}: ${avg_price:,.2f}")

            # Optional: Show a simple trend if data spans multiple years
            year_avg = town_data.groupby('List Year')['Sale Amount'].mean()
            if len(year_avg) > 1:
                print(f"\nHistorical yearly averages for {town}:")
                for year, price in year_avg.items():
                    print(f"{year}: ${price:,.2f}")

# Connect the widgets to the callback function
town_dropdown.observe(predict_and_display, names='value')
year_slider.observe(predict_and_display, names='value')
property_type_dropdown.observe(predict_and_display, names='value')

# Layout the widgets
input_widgets = widgets.VBox([town_dropdown, year_slider, property_type_dropdown])
app = widgets.VBox([input_widgets, output])

# Display the app
display(app)

# Initialize with a prediction
predict_and_display(None)